# Actividad Práctica · Semana 7
## Herramientas y lenguajes (SQL, NoSQL, Python)
### 🌱 Caso: Gestión del Riego Agroindustrial basada en Datos

```yaml
# CONFIG
FULL_NAME: Isabella Ramos Betancourt
GITHUB_USER: Isarb-21
```

**Isabella Ramos Betancourt**  
Facultad de Ingeniería, Corporación Universitaria del Huila (CORHUILA)  
Ciencia de Datos (Cód. 69109) [Pénsum 40D, Grupo 1]  
Profesor: Jesús Ariel Gonzáles Bonilla  
Octubre de 2026

---
### 📌 Objetivo del Cuaderno
Implementar y ejecutar de forma interactiva **3 consultas SQL** (filtro con `WHERE`, combinación con `JOIN`, y agregación con `GROUP BY`) sobre el modelo relacional del sistema de riego agroindustrial diseñado en la Semana 6, y **replicar la consulta analítica en pandas**, comparando la sintaxis y ventajas de ambos entornos.

## 1. Introducción y Contexto del Caso
Como señalan Silberschatz et al. (2020), SQL es el lenguaje declarativo estándar para bases de datos relacionales, permitiendo extraer y consolidar registros con alta eficiencia. Por su parte, la librería **pandas** de Python ofrece una estructura de manipulación en memoria (*DataFrames*) que permite replicar estas operaciones e integrarlas en pipelines de aprendizaje automático (McKinney, 2022).

En esta actividad trabajamos sobre las tablas del modelo del proyecto:
- **`lote`**: Información agronómica (área, tipo de suelo, cultivo, rendimiento final en toneladas).
- **`tecnico`**: Personal responsable de la operación de campo.
- **`registro_riego`**: Eventos individuales de aplicación de agua (fecha, lote, técnico, volumen en m³, duración en minutos).

In [1]:
import sqlite3
import pandas as pd

# Crear base de datos relacional en memoria
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 1. Crear tabla Lote
cursor.execute('''
CREATE TABLE lote (
    id_lote VARCHAR(10) PRIMARY KEY,
    nombre VARCHAR(50) NOT NULL,
    area_ha DECIMAL(5,2),
    tipo_suelo VARCHAR(30),
    cultivo VARCHAR(40),
    rendimiento_ton DECIMAL(6,2)
);
''')

# 2. Crear tabla Tecnico
cursor.execute('''
CREATE TABLE tecnico (
    id_tecnico VARCHAR(10) PRIMARY KEY,
    nombre VARCHAR(60) NOT NULL,
    especialidad VARCHAR(40),
    telefono VARCHAR(15)
);
''')

# 3. Crear tabla Registro_Riego
cursor.execute('''
CREATE TABLE registro_riego (
    id_riego INTEGER PRIMARY KEY,
    id_lote VARCHAR(10),
    id_tecnico VARCHAR(10),
    fecha DATE NOT NULL,
    volumen_agua_m3 DECIMAL(6,2) NOT NULL,
    duracion_min INTEGER,
    FOREIGN KEY (id_lote) REFERENCES lote(id_lote),
    FOREIGN KEY (id_tecnico) REFERENCES tecnico(id_tecnico)
);
''')

print("✓ Tablas creadas exitosamente en SQLite en memoria.")

✓ Tablas creadas exitosamente en SQLite en memoria.


In [2]:
# Insertar lotes
lotes_data = [
    ('LOT-001', 'Lote Norte - Maíz', 5.2, 'Franco-arcilloso', 'Maíz dulce', 28.5),
    ('LOT-002', 'Lote Sur - Tomate', 3.8, 'Arenoso', 'Tomate chonto', 42.0),
    ('LOT-003', 'Lote Este - Café', 6.0, 'Franco-limoso', 'Café castillo', 18.2)
]
cursor.executemany("INSERT INTO lote VALUES (?, ?, ?, ?, ?, ?);", lotes_data)

# Insertar técnicos
tecnicos_data = [
    ('TEC-01', 'Carlos Mendoza', 'Riego por goteo', '3101234567'),
    ('TEC-02', 'María Ospina', 'Monitoreo de humedad', '3209876543')
]
cursor.executemany("INSERT INTO tecnico VALUES (?, ?, ?, ?);", tecnicos_data)

# Insertar eventos de riego
riegos_data = [
    (1, 'LOT-001', 'TEC-01', '2026-09-01', 3.4, 45),
    (2, 'LOT-001', 'TEC-02', '2026-09-03', 2.8, 35),
    (3, 'LOT-001', 'TEC-01', '2026-09-05', 4.1, 55),
    (4, 'LOT-002', 'TEC-02', '2026-09-01', 5.0, 60),
    (5, 'LOT-002', 'TEC-01', '2026-09-04', 4.8, 58),
    (6, 'LOT-003', 'TEC-02', '2026-09-02', 1.9, 30),
    (7, 'LOT-003', 'TEC-01', '2026-09-06', 2.2, 35)
]
cursor.executemany("INSERT INTO registro_riego VALUES (?, ?, ?, ?, ?, ?);", riegos_data)
conn.commit()

print("✓ Datos de prueba cargados:")
print(f"  - Lotes: {len(lotes_data)}")
print(f"  - Técnicos: {len(tecnicos_data)}")
print(f"  - Eventos de riego: {len(riegos_data)}")

✓ Datos de prueba cargados:
  - Lotes: 3
  - Técnicos: 2
  - Eventos de riego: 7


## 2. Consultas SQL

### Consulta 1: Filtro con `WHERE`
**Pregunta operativa:** ¿En qué fechas se aplicó más de 3.0 m³ de agua en el Lote Norte (`LOT-001`)?  
Permite auditar riegos con alto consumo hídrico para detectar posibles excesos o sobrecostos.

In [3]:
sql_1 = '''
SELECT fecha, volumen_agua_m3, duracion_min
FROM registro_riego
WHERE id_lote = 'LOT-001'
  AND volumen_agua_m3 > 3.0
ORDER BY fecha;
'''

df_c1 = pd.read_sql_query(sql_1, conn)
print("Resultado Consulta 1 (Filtro WHERE):")
df_c1

Resultado Consulta 1 (Filtro WHERE):
        fecha  volumen_agua_m3  duracion_min
0  2026-09-01              3.4            45
1  2026-09-05              4.1            55


**Interpretación:**  
La consulta filtra los registros de `registro_riego` cumpliendo simultáneamente la condición de lote (`LOT-001`) y el umbral de consumo (`> 3.0 m³`). Se ordenan cronológicamente con `ORDER BY fecha`.

### Consulta 2: Combinación con `JOIN`
**Pregunta operativa:** ¿Qué técnico ejecutó cada riego y cuánta agua aplicó en cada lote?  
Permite auditar la responsabilidad del personal de campo vinculando la clave foránea `id_tecnico` con su nombre real.

In [4]:
sql_2 = '''
SELECT t.nombre AS tecnico,
       r.fecha,
       r.id_lote,
       r.volumen_agua_m3,
       r.duracion_min
FROM registro_riego r
JOIN tecnico t ON r.id_tecnico = t.id_tecnico
ORDER BY r.fecha;
'''

df_c2 = pd.read_sql_query(sql_2, conn)
print("Resultado Consulta 2 (JOIN):")
df_c2

Resultado Consulta 2 (JOIN):
          tecnico       fecha  id_lote  volumen_agua_m3  duracion_min
0  Carlos Mendoza  2026-09-01  LOT-001              3.4            45
1    María Ospina  2026-09-01  LOT-002              5.0            60
2    María Ospina  2026-09-02  LOT-003              1.9            30
3    María Ospina  2026-09-03  LOT-001              2.8            35
4  Carlos Mendoza  2026-09-04  LOT-002              4.8            58
5  Carlos Mendoza  2026-09-05  LOT-001              4.1            55
6  Carlos Mendoza  2026-09-06  LOT-003              2.2            35


**Interpretación:**  
El `INNER JOIN` asocia cada evento con el registro correspondiente de la tabla `tecnico`. Reemplaza el identificador crudo (`TEC-01`) por el nombre legible del operario, facilitando la auditoría de campo.

### Consulta 3: Agregación con `GROUP BY` y Cálculo de Eficiencia
**Pregunta operativa:** ¿Cuánta agua total consumió cada lote, cuántos riegos tuvo y cuál fue su eficiencia hídrica (toneladas cosechadas por m³ aplicado)?  
Permite responder directamente la pregunta central de investigación del proyecto agroindustrial (FAO, 2026).

In [5]:
sql_3 = '''
SELECT l.nombre AS lote,
       l.cultivo,
       l.rendimiento_ton,
       SUM(r.volumen_agua_m3) AS agua_total_m3,
       COUNT(r.id_riego) AS num_riegos,
       ROUND(l.rendimiento_ton / NULLIF(SUM(r.volumen_agua_m3), 0), 3) AS eficiencia_ton_m3
FROM lote l
JOIN registro_riego r ON l.id_lote = r.id_lote
GROUP BY l.id_lote, l.nombre, l.cultivo, l.rendimiento_ton
ORDER BY eficiencia_ton_m3 DESC;
'''

df_c3 = pd.read_sql_query(sql_3, conn)
print("Resultado Consulta 3 (GROUP BY + Agregaciones):")
df_c3

Resultado Consulta 3 (GROUP BY + Agregaciones):
                 lote        cultivo  rendimiento_ton  agua_total_m3  num_riegos  eficiencia_ton_m3
0   Lote Sur - Tomate  Tomate chonto             42.0            9.8           2              4.286
1    Lote Este - Café  Café castillo             18.2            4.1           2              4.439
2  Lote Norte - Maíz    Maíz dulce             28.5           10.3           3              2.767


## 3. Réplica de la Consulta 3 en Python con pandas
A continuación replicamos la lógica completa de la Consulta 3 usando las funciones nativas de **pandas**:
1. `pd.read_sql_query` para cargar DataFrames base.
2. `pd.merge()` equivalente al `JOIN`.
3. `groupby().agg()` equivalente a las cláusulas `GROUP BY`, `SUM` y `COUNT`.
4. Creación vectorizada de la columna `eficiencia_ton_m3`.

In [6]:
# Extraer las tablas a DataFrames de pandas
df_lote = pd.read_sql_query("SELECT * FROM lote;", conn)
df_riego = pd.read_sql_query("SELECT * FROM registro_riego;", conn)

# 1. Cruzar DataFrames (JOIN)
df_combinado = pd.merge(df_riego, df_lote, on="id_lote", how="inner")

# 2. Agrupación y cálculo de agregaciones (GROUP BY + SUM + COUNT)
resumen_pandas = (
    df_combinado
    .groupby(["id_lote", "nombre", "cultivo", "rendimiento_ton"])
    .agg(
        agua_total_m3=("volumen_agua_m3", "sum"),
        num_riegos=("id_riego", "count")
    )
    .reset_index()
)

# 3. Calcular columna de eficiencia hídrica (ton / m3)
resumen_pandas["eficiencia_ton_m3"] = (
    resumen_pandas["rendimiento_ton"] / resumen_pandas["agua_total_m3"]
).round(3)

# 4. Ordenar descendente por eficiencia
resumen_pandas = resumen_pandas.sort_values("eficiencia_ton_m3", ascending=False)

print("Resultado en pandas (idéntico a la Consulta 3 de SQL):")
resumen_pandas[["nombre", "cultivo", "rendimiento_ton", "agua_total_m3", "num_riegos", "eficiencia_ton_m3"]]

Resultado en pandas (idéntico a la Consulta 3 de SQL):
              nombre        cultivo  rendimiento_ton  agua_total_m3  num_riegos  eficiencia_ton_m3
2   Lote Este - Café  Café castillo             18.2            4.1           2              4.439
1  Lote Sur - Tomate  Tomate chonto             42.0            9.8           2              4.286
0  Lote Norte - Maíz     Maíz dulce             28.5           10.3           3              2.767


## 4. Comparación Técnica: SQL vs Python (pandas)

| Criterio | SQL (Motor Relacional) | Python / pandas (En memoria) |
|---|---|---|
| **Paradigma** | Declarativo (*qué* datos traer). | Procedimental / Funcional (*cómo* transformarlos). |
| **Ubicación de cómputo** | En el servidor de base de datos (DBMS). | En la memoria RAM de la máquina cliente / notebook. |
| **Ventaja principal** | Optimizado con índices y claves para millones de filas. | Flexibilidad para crear visualizaciones, features de ML y pipelines. |
| **Casos de uso en el caso** | Extracción inicial de bitácoras y telemetría histórica. | Limpieza de datos (Semana 9), modelado predictivo y gráficos. |

## 5. Conclusiones y Referencias

### Conclusiones:
1. **Complementariedad de herramientas:** SQL y pandas no compiten; se complementan. SQL es la herramienta óptima para filtrar y agregar volúmenes masivos antes de transmitirlos por la red, mientras que pandas es ideal para la ingeniería de variables y análisis exploratorio fino.
2. **Eficiencia hídrica:** La métrica consolidada en la Consulta 3 (`eficiencia_ton_m3`) demuestra cómo el Lote Este (Café) y el Lote Sur (Tomate) superaron los 4.2 ton/m³, permitiendo a la empresa priorizar parcelas con mejor retorno agronómico.
3. **Reproducibilidad:** El uso de SQLite en memoria dentro del cuaderno interactivo permite verificar la sintaxis SQL de forma portátil sin depender de servidores externos.

---

### Referencias Bibliográficas:
* Food and Agriculture Organization of the United Nations (FAO). (2026). *Water data and resource assessment*. https://www.fao.org/land-water/water/water-data-and-resource-assessment
* IBM. (2024). *SQL vs. Python: What is the difference?* IBM Think. https://www.ibm.com/think/topics/sql-vs-python
* McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
* Silberschatz, A., Korth, H. F., & Sudarshan, S. (2020). *Database System Concepts* (7th ed.). McGraw-Hill.